<a href="https://colab.research.google.com/github/rmcN7/orion-t2/blob/main/code/opensky_pull.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [67]:
import requests


In [68]:
import time
import pandas as pd
from google.colab import userdata

client_id = userdata.get("OPENSKY_CLIENT_ID").strip()
client_secret = userdata.get("OPENSKY_CLIENT_SECRET").strip()
token_url = "https://auth.opensky-network.org/auth/realms/opensky-network/protocol/openid-connect/token"

def get_token():
    r = requests.post(token_url, data={
        "grant_type": "client_credentials",
        "client_id": client_id,
        "client_secret": client_secret,
    }, timeout=15)
    return r.json()["access_token"]

columns = ["icao24", "callsign", "origin_country", "time_position", "last_contact",
           "longitude", "latitude", "baro_altitude", "on_ground", "velocity",
           "true_track", "vertical_rate", "sensors", "geo_altitude", "squawk",
           "spi", "position_source", "category"]

params = {"lamin": 40, "lomin": 27, "lamax": 47, "lomax": 42}

all_pulls = []
num_pulls = 6
wait_seconds = 300  # 5 min between pulls, 6 pulls = 30 min window

for i in range(num_pulls):
    token = get_token()
    headers = {"Authorization": f"Bearer {token}"}
    resp = requests.get("https://opensky-network.org/api/states/all", headers=headers, params=params, timeout=15)
    data = resp.json()
    if data.get("states"):
        df_pull = pd.DataFrame(data["states"], columns=columns[:len(data["states"][0])])
        df_pull["pulled_at"] = data["time"]
        all_pulls.append(df_pull)
        print(f"Pull {i+1}: {len(df_pull)} aircraft")
    else:
        print(f"Pull {i+1}: no aircraft")
    if i < num_pulls - 1:
        time.sleep(wait_seconds)

df_tracks = pd.concat(all_pulls, ignore_index=True)
print(df_tracks.shape)

Pull 1: 107 aircraft
Pull 2: 106 aircraft
Pull 3: 102 aircraft
Pull 4: 104 aircraft
Pull 5: 104 aircraft
Pull 6: 102 aircraft
(625, 18)


In [69]:
df_tracks.to_csv("opensky_raw_sample.csv", index=False)

In [70]:
print(df_tracks.isna().sum())

icao24               0
callsign             0
origin_country       0
time_position        0
last_contact         0
longitude            0
latitude             0
baro_altitude       16
on_ground            0
velocity             0
true_track           0
vertical_rate       16
sensors            625
geo_altitude        20
squawk             184
spi                  0
position_source      0
pulled_at            0
dtype: int64


In [71]:
df_tracks = df_tracks.sort_values(["icao24", "pulled_at"])
busiest = df_tracks["icao24"].value_counts().index[0]
df_tracks[df_tracks["icao24"] == busiest]

,icao24,callsign,origin_country,time_position,last_contact,longitude,latitude,baro_altitude,on_ground,velocity,true_track,vertical_rate,sensors,geo_altitude,squawk,spi,position_source,pulled_at
99,3c4581,BOX544,Germany,1783318431,1783318494,27.828,44.076,11277.6,False,259.74,122.46,0.33,None,11551.92,7667,False,0,1783318496
207,3c4581,BOX544,Germany,1783318431,1783318797,27.828,44.076,11277.6,False,259.74,122.46,0.33,None,11551.92,7667,False,0,1783318799
309,3c4581,BOX544,Germany,1783318431,1783319098,27.828,44.076,11277.6,False,259.74,122.46,0.33,None,11551.92,7667,False,0,1783319101
414,3c4581,BOX544,Germany,1783318431,1783319398,27.828,44.076,11277.6,False,259.74,122.46,0.33,None,11551.92,7667,False,0,1783319400
516,3c4581,BOX544,Germany,1783318431,1783319690,27.828,44.076,11277.6,False,259.74,122.46,0.33,None,11551.92,7667,False,0,1783319704
619,3c4581,BOX544,Germany,1783318431,1783319852,27.828,44.076,11277.6,False,259.74,122.46,0.33,None,11551.92,7667,False,0,1783320002
